# TP3 - Attribution : qualite des clients par canal

In [1]:
import pandas as pd

transactions = pd.read_csv("../Lumina & Co - CRM/transactions.csv")
touchpoints = pd.read_csv("../Lumina & Co - CRM/touchpoints.csv")

## Etape 1 : reconstruire le RFM (comme dans segmentation.ipynb)

In [2]:
# on refait le meme calcul RFM que dans segmentation.ipynb
transactions = transactions[transactions['customer_id'].notna()]
transactions['line_total'] = transactions['quantity'] * transactions['unit_price']
transactions['invoice_date'] = pd.to_datetime(transactions['invoice_date'])
snapshot_date = transactions['invoice_date'].max()

recency = (snapshot_date - transactions.groupby('customer_id')['invoice_date'].max()).dt.days.rename('recency')
frequency = transactions.groupby('customer_id')['invoice_id'].nunique().rename('frequency')
monetary = transactions.groupby('customer_id')['line_total'].sum().rename('monetary')

rfm = pd.concat([recency, frequency, monetary], axis=1).reset_index()
rfm.head()

,customer_id,recency,frequency,monetary
0,12346.0,325,17,701.46
1,12347.0,1,8,7740.74
2,12348.0,74,5,1561.92
3,12349.0,18,5,5342.41
4,12350.0,309,1,554.75


In [3]:
rfm['r_score'] = pd.qcut(rfm['recency'].rank(method='first'), 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['m_score'] = pd.qcut(rfm['monetary'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['rfm_score'] = rfm['r_score'].astype(str) + rfm['f_score'].astype(str) + rfm['m_score'].astype(str)

In [4]:
def label_segment(row):
    if row['rfm_score'] == '555':
        return 'Champions'
    if row['rfm_score'] == '111':
        return 'Perdus'
    if row['r_score'] <= 2 and row['f_score'] >= 4 and row['m_score'] >= 4:
        return 'A risque'
    return 'Autres'

rfm['segment'] = rfm.apply(label_segment, axis=1)
rfm['segment'].value_counts()

segment
Autres       41142
Perdus        3134
Champions     3068
A risque      2058
Name: count, dtype: int64

**A retenir :** on refait exactement le meme RFM que dans segmentation.ipynb pour pouvoir croiser les segments avec les canaux d'acquisition. 6,2% de la base sont Champions, 6,3% Perdus, 4,2% A risque, le reste en Autres.

## Etape 2 : quel canal ramene quel type de client (modele dernier clic)

In [5]:
# on garde les touchpoints marques comme "dernier clic avant conversion"
conv = touchpoints[touchpoints['is_last_before_conversion'] == 1].copy()

# un client peut convertir plusieurs fois sur un meme canal, on le compte 1 seule fois par canal
conv_unique = conv.drop_duplicates(subset=['customer_id', 'channel'])

merged = conv_unique.merge(rfm[['customer_id', 'segment']], on='customer_id', how='left')
merged.head()

,touchpoint_id,customer_id,invoice_id,journey_id,campaign_name,timestamp,channel,position,n_touchpoints,click,converted,cost,is_last_before_conversion,segment
0,2,12346,541431.0,conv_541431,Summer_Sale,2025-08-08 19:43:30.750000041,retargeting,4,4,1,1,0.0128,1,A risque
1,44,12347,549222.0,conv_549222,Black_Friday,2025-10-26 19:12:31.370521550,affiliate,5,5,1,1,172.5943,1,Champions
2,54,12347,562032.0,conv_562032,Valentine,2026-02-20 18:33:36.670668285,retargeting,5,5,1,1,0.0188,1,Champions
3,61,12348,539318.0,conv_539318,Summer_Sale,2025-07-07 02:50:07.719223540,retargeting,3,3,1,1,0.0197,1,Autres
4,71,12348,541998.0,conv_541998,Summer_Sale,2025-08-15 18:37:22.311613444,direct,5,5,1,1,0.0000,1,Autres


In [6]:
crosstab = pd.crosstab(merged['channel'], merged['segment'], normalize='index') * 100
crosstab = crosstab.round(1)
crosstab

segment,A risque,Autres,Champions
channel,,,
affiliate,1.4,83.2,15.4
direct,0.5,81.8,17.8
email,0.6,81.0,18.3
retargeting,0.6,82.4,17.0
search_paid,1.0,79.5,19.5


In [7]:
merged['channel'].value_counts()

channel
affiliate      16693
retargeting    14839
direct         12037
email          10639
search_paid    10323
Name: count, dtype: int64

**A retenir :** affiliate ramene le plus de clients uniques convertis mais la plus faible part de Champions (15,4%). Search_paid ramene le moins de clients mais la part de Champions la plus elevee (19,5%). Le canal le plus efficace en volume n'est pas forcement le plus qualitatif en valeur client -> a prendre en compte dans l'allocation budgetaire, pas juste le CPA/volume brut.